In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging

import pandas as pd

from drzavni_posao import (
    MediaStore,
    MlflowTracker,
    Stage,
    evaluate_run,
)
from drzavni_posao.metrics import (
    assignment_scores,
    character_table,
    confusion_table,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.compare_assignment")


In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
TRANSCRIPT_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)
LABEL_DIR = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels"
)
EVAL_CACHE = None

# How many recent configuration runs that mention character_assigner to inspect.
CANDIDATE_LIMIT = 50

tracker = MlflowTracker(EXPERIMENT_NAME)
media = MediaStore(AUDIO_PATHS, TRANSCRIPT_PATH, trim_start=25, trim_end=7)
ALL_EPISODES = {episode.key for episode in media.episodes()}
len(ALL_EPISODES)


In [ ]:
def assignment_complete_on_all(run_id: str) -> tuple[bool, int, int]:
    """True when every corpus episode finished character assignment."""
    states = tracker.episode_run_states(run_id)
    done = {
        key
        for key, state in states.items()
        if Stage.CHARACTER_ASSIGNER in state.completed_stages
    }
    return ALL_EPISODES <= done, len(done), len(states)


def run_summary(run) -> dict:
    config = tracker._configuration(run.info.run_id)
    assigner = config.get("character_assigner") or {}
    embedder = assigner.get("embedder") or {}
    transcription = config.get("transcription") or {}
    samples = assigner.get("speaker_samples") or {}
    return {
        "run_id": run.info.run_id,
        "name": run.info.run_name,
        "started": datetime.fromtimestamp(
            run.info.start_time / 1000, tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M"),
        "stages": run.data.tags.get("stages"),
        "gallery_tag": run.data.tags.get("gallery"),
        "experiment_tag": run.data.tags.get("experiment"),
        "source_run_id": run.data.tags.get("source.run_id"),
        "asr_model": transcription.get("model_name"),
        "min_similarity": assigner.get("min_similarity"),
        "min_segment_seconds": embedder.get("min_segment_seconds"),
        "gallery_speakers": len(samples),
        "gallery_clips": sum(len(clips) for clips in samples.values()),
    }


experiment = tracker.client.get_experiment_by_name(EXPERIMENT_NAME)
candidates = tracker.client.search_runs(
    [experiment.experiment_id],
    (
        "tags.run_level = 'configuration' "
        "AND tags.stages LIKE '%character_assigner%'"
    ),
    order_by=["attributes.start_time DESC"],
    max_results=CANDIDATE_LIMIT,
)

rows = []
for run in candidates:
    ok, assigned, children = assignment_complete_on_all(run.info.run_id)
    row = run_summary(run)
    row.update(
        {
            "assigned_episodes": assigned,
            "child_episodes": children,
            "complete_on_all": ok,
        }
    )
    rows.append(row)

candidates_df = pd.DataFrame(rows)
display(candidates_df)

ASSIGNMENT_RUNS = candidates_df[candidates_df["complete_on_all"]].copy()
ASSIGNMENT_RUNS


In [ ]:
evaluated = {}
score_rows = []

for row in ASSIGNMENT_RUNS.itertuples(index=False):
    label = f"{row.name} ({row.run_id[:8]})"
    logger.info("Evaluating %s", label)
    segments = evaluate_run(tracker, row.run_id, LABEL_DIR, EVAL_CACHE)
    evaluated[label] = segments
    scores = assignment_scores(segments)
    score_rows.append(
        {
            "label": label,
            "run_id": row.run_id,
            "name": row.name,
            "started": row.started,
            "asr_model": row.asr_model,
            "min_similarity": row.min_similarity,
            "min_segment_seconds": row.min_segment_seconds,
            "gallery_speakers": row.gallery_speakers,
            "gallery_clips": row.gallery_clips,
            "gallery_tag": row.gallery_tag,
            **scores,
        }
    )

comparison = (
    pd.DataFrame(score_rows)
    .set_index("label")
    .sort_values("accuracy", ascending=False)
)
comparison.round(3)


In [ ]:
per_episode = pd.DataFrame(
    {
        label: {
            key: assignment_scores(episode)["accuracy"]
            for key, episode in segments.groupby("episode_key")
        }
        for label, segments in evaluated.items()
    }
).sort_index(key=lambda idx: idx.map(lambda k: int(k) if str(k).isdigit() else k))

display(per_episode.round(3))
per_episode.round(3).describe()


In [ ]:
# Pick one row label from the comparison table to inspect further.
DRILLDOWN = comparison.index[0]

scored = evaluated[DRILLDOWN]
episode_scores = pd.DataFrame(
    {
        key: assignment_scores(episode)
        for key, episode in scored.groupby("episode_key")
    }
).T.sort_values("accuracy")

display(episode_scores.round(3))
display(confusion_table(scored).round(1))
display(character_table(scored).round(3))
scored[~scored["correct"]].sort_values("seconds", ascending=False).head(20)
